##### Imports

In [1]:
# pip install llama-index-lms-ollama
# pip install llama-index-embeddings-ollama

In [2]:
from pathlib import Path
import csv
import json
import re
from collections import defaultdict
from difflib import SequenceMatcher
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

/home/cpanagiotop/CaseBundleGen_Eval/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = Ollama(model="gemma3:12b", request_timeout=360.0)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20147.40it/s]


In [4]:
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text


def exact_match(reference: str, prediction: str) -> bool:
    return normalize_text(reference) == normalize_text(prediction)


def fuzzy_match(reference: str, prediction: str, threshold: float = 0.80) -> bool:
    ref_norm = normalize_text(reference)
    pred_norm = normalize_text(prediction)
    if not ref_norm or not pred_norm:
        return False
    return SequenceMatcher(None, ref_norm, pred_norm).ratio() >= threshold


def response_to_text(response) -> str:
    if response is None:
        return ""
    if hasattr(response, "text"):
        return response.text
    if hasattr(response, "response"):
        return response.response
    return str(response)


def list_case_ids():
    return sorted([p.name for p in Path("cases").iterdir() if p.is_dir()])


def load_qa(case_id: str):
    qa_path = Path("quesNAs") / case_id / "qa.json"
    with qa_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def build_case_index(case_id: str):
    case_path = Path("cases") / case_id
    documents = SimpleDirectoryReader(str(case_path)).load_data()
    return VectorStoreIndex.from_documents(documents)


def judge_answer(question: str, expected: str, predicted: str, llm=None) -> tuple[str, str, str]:
    if llm is None:
        llm = Settings.llm
    prompt = (
        "You are a precise QA judge. Compare the expected answer and the model answer "
        "for the question below. Return only one word: EXACT, PARTIAL, or WRONG. "
        "Do not add any explanation, punctuation, or extra text.\n\n"
        f"Question: {question}\n"
        f"Expected answer: {expected}\n"
        f"Model answer: {predicted}\n"
    )
    response = llm.complete(prompt=prompt)
    raw_text = response_to_text(response).strip()
    normalized_label = normalize_text(raw_text).upper()
    if normalized_label == "EXACT":
        return "EXACT", raw_text, normalized_label
    if normalized_label == "PARTIAL":
        return "PARTIAL", raw_text, normalized_label
    if normalized_label == "WRONG":
        return "WRONG", raw_text, normalized_label
    if "exact" in normalized_label:
        return "EXACT", raw_text, normalized_label
    if "partial" in normalized_label:
        return "PARTIAL", raw_text, normalized_label
    if "wrong" in normalized_label:
        return "WRONG", raw_text, normalized_label
    return "WRONG", raw_text, normalized_label


def evaluate_case(case_id: str, index, use_judge: bool = True):
    qa_pairs = load_qa(case_id)
    query_engine = index.as_query_engine()

    case_results = []
    for pair in qa_pairs:
        question = pair.get("question")
        expected = pair.get("answer")
        response = query_engine.query(question)
        predicted = response_to_text(response)

        is_exact = exact_match(expected, predicted)
        is_similar = fuzzy_match(expected, predicted)
        judge_label, judge_raw, judge_normalized = (judge_answer(question, expected, predicted) if use_judge else (None, None, None))

        case_results.append({
            "case_id": case_id,
            "question": question,
            "expected_answer": expected,
            "predicted_answer": predicted,
            "exact_match": is_exact,
            "similar_match": is_similar,
            "judge_label": judge_label,
            "judge_raw": judge_raw,
            "judge_normalized": judge_normalized,
        })

    return case_results


def evaluate_all_cases(use_judge: bool = True):
    all_results = []
    for case_id in list_case_ids():
        print(f"Building index for {case_id}...")
        index = build_case_index(case_id)
        print(f"Evaluating {case_id}...")
        case_results = evaluate_case(case_id, index, use_judge=use_judge)
        all_results.extend(case_results)
    return all_results


def summarize_results(results):
    total = len(results)
    exact = sum(1 for r in results if r["exact_match"])
    similar = sum(1 for r in results if r["similar_match"])
    judge_exact = sum(1 for r in results if r.get("judge_label") == "EXACT")
    print(f"Total questions: {total}")
    print(f"Exact-match accuracy: {exact}/{total} = {exact/total:.2%}")
    print(f"Fuzzy-match accuracy: {similar}/{total} = {similar/total:.2%}")
    print(f"Judge EXACT accuracy: {judge_exact}/{total} = {judge_exact/total:.2%}")
    return {
        "total_questions": total,
        "exact_match_count": exact,
        "fuzzy_match_count": similar,
        "judge_exact_count": judge_exact,
        "exact_match_accuracy": exact / total if total else 0.0,
        "fuzzy_match_accuracy": similar / total if total else 0.0,
        "judge_exact_accuracy": judge_exact / total if total else 0.0,
    }


def summarize_by_case(results):
    cases = defaultdict(list)
    for r in results:
        cases[r["case_id"]].append(r)

    per_case = {}
    for case_id, rows in cases.items():
        total = len(rows)
        exact = sum(1 for r in rows if r["exact_match"])
        similar = sum(1 for r in rows if r["similar_match"])
        judge_exact = sum(1 for r in rows if r.get("judge_label") == "EXACT")
        per_case[case_id] = {
            "total_questions": total,
            "exact_match_count": exact,
            "fuzzy_match_count": similar,
            "judge_exact_count": judge_exact,
            "exact_match_accuracy": exact / total if total else 0.0,
            "fuzzy_match_accuracy": similar / total if total else 0.0,
            "judge_exact_accuracy": judge_exact / total if total else 0.0,
        }

    exact_rates = [stats["exact_match_accuracy"] for stats in per_case.values()]
    fuzzy_rates = [stats["fuzzy_match_accuracy"] for stats in per_case.values()]
    judge_rates = [stats["judge_exact_accuracy"] for stats in per_case.values()]

    if exact_rates:
        print("Per-case exact match rates:")
        for case_id, stats in per_case.items():
            print(f" - {case_id}: {stats['exact_match_accuracy']:.2%}")
        print(f"Min exact-match accuracy: {min(exact_rates):.2%}")
        print(f"Max exact-match accuracy: {max(exact_rates):.2%}")
        print(f"Min fuzzy-match accuracy: {min(fuzzy_rates):.2%}")
        print(f"Max fuzzy-match accuracy: {max(fuzzy_rates):.2%}")
        print(f"Min judge EXACT accuracy: {min(judge_rates):.2%}")
        print(f"Max judge EXACT accuracy: {max(judge_rates):.2%}")

    return {
        "per_case": per_case,
        "min_exact_match_accuracy": min(exact_rates) if exact_rates else 0.0,
        "max_exact_match_accuracy": max(exact_rates) if exact_rates else 0.0,
        "min_fuzzy_match_accuracy": min(fuzzy_rates) if fuzzy_rates else 0.0,
        "max_fuzzy_match_accuracy": max(fuzzy_rates) if fuzzy_rates else 0.0,
        "min_judge_exact_accuracy": min(judge_rates) if judge_rates else 0.0,
        "max_judge_exact_accuracy": max(judge_rates) if judge_rates else 0.0,
    }


def write_results_csv(results, path="evaluation_results.csv"):
    if not results:
        print("No results to write.")
        return
    fieldnames = [
        "case_id",
        "question",
        "expected_answer",
        "predicted_answer",
        "exact_match",
        "similar_match",
        "judge_label",
        "judge_raw",
        "judge_normalized",
    ]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in results:
            writer.writerow(row)
    print(f"Written results to {path}")


# Example run:
# results = evaluate_all_cases()
# summary = summarize_results(results)
# case_summary = summarize_by_case(results)
# write_results_csv(results, "case_benchmark_results.csv")


In [6]:
# Run the full benchmark, print summaries, and export to CSV.
results = evaluate_all_cases()
summary = summarize_results(results)
case_summary = summarize_by_case(results)
write_results_csv(results, "case_benchmark_results.csv")
print(summary)
print(case_summary)


Building index for case_001...
Evaluating case_001...
Building index for case_002...
Evaluating case_002...
Building index for case_003...
Evaluating case_003...
Building index for case_004...
Evaluating case_004...
Building index for case_005...
Evaluating case_005...
Building index for case_006...
Evaluating case_006...
Building index for case_007...
Evaluating case_007...
Building index for case_008...
Evaluating case_008...
Building index for case_009...
Evaluating case_009...
Building index for case_010...
Evaluating case_010...
Total questions: 90
Exact-match accuracy: 50/90 = 55.56%
Fuzzy-match accuracy: 51/90 = 56.67%
Judge EXACT accuracy: 73/90 = 81.11%
Per-case exact match rates:
 - case_001: 50.00%
 - case_002: 75.00%
 - case_003: 80.00%
 - case_004: 50.00%
 - case_005: 37.50%
 - case_006: 50.00%
 - case_007: 50.00%
 - case_008: 25.00%
 - case_009: 80.00%
 - case_010: 50.00%
Min exact-match accuracy: 25.00%
Max exact-match accuracy: 80.00%
Min fuzzy-match accuracy: 25.00%
Ma